# 112. Cross-Modal Retrieval: Finding Across Modalities

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amerob/ultimate-prompt-engineering-playbook/blob/main/notebooks/13-multi-modal/112_cross_modal_retrieval.ipynb)

**Category:** 13 - Multi-Modal Techniques  
**Technique #:** 112  
**Difficulty:** Advanced

## 📖 Description

Cross-Modal Retrieval enables searching for content across different modalities - finding images with text queries, retrieving text based on images, or matching audio to visual content. This technique uses shared embedding spaces to bridge the gap between different types of data.

### When to Use:
- Image search with text descriptions
- Finding relevant text for an image
- Content recommendation across modalities
- Visual question answering systems
- Multi-modal content organization

## 🔧 How It Works

```
┌─────────────────────────────────────────────────────────────┐
│              CROSS-MODAL RETRIEVAL ARCHITECTURE              │
└─────────────────────────────────────────────────────────────┘

                    ┌─────────────────────┐
                    │   SHARED EMBEDDING  │
                    │       SPACE         │
                    └──────────┬──────────┘
                               │
           ┌───────────────────┼───────────────────┐
           │                   │                   │
           ▼                   ▼                   ▼
    ┌──────────────┐    ┌──────────────┐    ┌──────────────┐
    │   Text       │    │   Image      │    │   Audio      │
    │   Encoder    │    │   Encoder    │    │   Encoder    │
    └──────┬───────┘    └──────┬───────┘    └──────┬───────┘
           │                   │                   │
           ▼                   ▼                   ▼
    ┌──────────────┐    ┌──────────────┐    ┌──────────────┐
    │  "red car"   │    │   [Image]    │    │   [Audio]    │
    │  ────────▶   │    │   ────────▶  │    │   ────────▶  │
    │  [0.2, 0.8,  │    │  [0.2, 0.8,  │    │  [0.2, 0.8,  │
    │   0.5, ...]  │    │   0.5, ...]  │    │   0.5, ...]  │
    └──────────────┘    └──────────────┘    └──────────────┘

    Query: "red car" ──▶ Find nearest neighbors ──▶ Returns images of red cars
```

### Key Concepts:
- **Shared Embedding Space**: Same vector space for all modalities
- **Contrastive Learning**: Training to align different modalities
- **Similarity Search**: Finding nearest neighbors across modalities
- **Zero-Shot Retrieval**: No modality-specific training needed

## 🛠️ Setup

In [ ]:
!pip install -q openai pillow requests numpy scikit-learn

In [ ]:
import os
from getpass import getpass
import base64
import requests
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

api_key = getpass("Enter your OpenAI API key: ")
os.environ["OPENAI_API_KEY"] = api_key

from openai import OpenAI
client = OpenAI()

## 💡 Basic Example

In [ ]:
# Text-to-Image retrieval using text embeddings
class SimpleCrossModalRetriever:
    """Simple cross-modal retrieval using text embeddings."""
    
    def __init__(self):
        self.items = []  # Stores (id, text_description, image_url, embedding)
    
    def add_item(self, item_id, text_description, image_url):
        """Add an item with both text and image."""
        # Generate embedding from text description
        response = client.embeddings.create(
            model="text-embedding-3-small",
            input=text_description
        )
        embedding = np.array(response.data[0].embedding)
        
        self.items.append({
            "id": item_id,
            "text": text_description,
            "image_url": image_url,
            "embedding": embedding
        })
    
    def text_search(self, query, top_k=3):
        """Search for items using text query."""
        # Generate query embedding
        response = client.embeddings.create(
            model="text-embedding-3-small",
            input=query
        )
        query_embedding = np.array(response.data[0].embedding).reshape(1, -1)
        
        # Calculate similarities
        item_embeddings = np.array([item["embedding"] for item in self.items])
        similarities = cosine_similarity(query_embedding, item_embeddings)[0]
        
        # Get top-k indices
        top_indices = np.argsort(similarities)[-top_k:][::-1]
        
        results = []
        for idx in top_indices:
            results.append({
                "id": self.items[idx]["id"],
                "text": self.items[idx]["text"],
                "image_url": self.items[idx]["image_url"],
                "similarity": similarities[idx]
            })
        
        return results
    
    def image_search(self, query_image_description, top_k=3):
        """Search for text using image description (simulated)."""
        # This simulates searching by converting image to description first
        return self.text_search(query_image_description, top_k)

# Initialize retriever
retriever = SimpleCrossModalRetriever()

# Add sample items
sample_items = [
    {
        "id": "1",
        "text": "A red sports car parked on a city street, sleek design with alloy wheels",
        "image_url": "https://images.unsplash.com/photo-1503376763036-066120622c74?w=400"
    },
    {
        "id": "2",
        "text": "A golden retriever dog playing in a green park on a sunny day",
        "image_url": "https://images.unsplash.com/photo-1633722715463-d30f4f325e24?w=400"
    },
    {
        "id": "3",
        "text": "Fresh Italian pasta dish with tomato sauce and basil on a white plate",
        "image_url": "https://images.unsplash.com/photo-1626844131082-256783844137?w=400"
    },
    {
        "id": "4",
        "text": "Modern laptop computer on a wooden desk with coffee cup nearby",
        "image_url": "https://images.unsplash.com/photo-1496181133206-80ce9b88a853?w=400"
    },
    {
        "id": "5",
        "text": "Beautiful sunset over ocean waves with orange and purple sky",
        "image_url": "https://images.unsplash.com/photo-1507525428034-b723cf961d3e?w=400"
    }
]

for item in sample_items:
    retriever.add_item(item["id"], item["text"], item["image_url"])

# Test text-to-image retrieval
print("CROSS-MODAL RETRIEVAL - BASIC EXAMPLE\n")
print("="*60 + "\n")

queries = [
    "fast vehicle",
    "cute animal pet",
    "delicious food",
    "work technology",
    "nature scenery"
]

for query in queries:
    print(f"Query: '{query}'")
    results = retriever.text_search(query, top_k=2)
    print("Top matches:")
    for r in results:
        print(f"  - {r['text'][:60]}... (similarity: {r['similarity']:.3f})")
    print()

## 🌍 Real-World Example

In [ ]:
# Real-world: E-commerce product search
class ProductSearchEngine:
    """E-commerce product search with cross-modal capabilities."""
    
    def __init__(self):
        self.products = []
    
    def add_product(self, product_id, name, description, category, price, image_url, attributes):
        """Add a product to the search index."""
        # Create rich text representation
        rich_text = f"""
        Product: {name}
        Category: {category}
        Description: {description}
        Attributes: {', '.join([f'{k}={v}' for k, v in attributes.items()])}
        """
        
        # Generate embedding
        response = client.embeddings.create(
            model="text-embedding-3-small",
            input=rich_text
        )
        embedding = np.array(response.data[0].embedding)
        
        self.products.append({
            "id": product_id,
            "name": name,
            "description": description,
            "category": category,
            "price": price,
            "image_url": image_url,
            "attributes": attributes,
            "embedding": embedding
        })
    
    def search(self, query, filters=None, top_k=5):
        """Search products with optional filters."""
        # Generate query embedding
        response = client.embeddings.create(
            model="text-embedding-3-small",
            input=query
        )
        query_embedding = np.array(response.data[0].embedding).reshape(1, -1)
        
        # Calculate similarities
        product_embeddings = np.array([p["embedding"] for p in self.products])
        similarities = cosine_similarity(query_embedding, product_embeddings)[0]
        
        # Apply filters if provided
        candidates = []
        for i, product in enumerate(self.products):
            if filters:
                skip = False
                for key, value in filters.items():
                    if key == "max_price" and product["price"] > value:
                        skip = True
                        break
                    if key == "category" and product["category"] != value:
                        skip = True
                        break
                if skip:
                    continue
            candidates.append((i, similarities[i]))
        
        # Sort by similarity and get top-k
        candidates.sort(key=lambda x: x[1], reverse=True)
        top_candidates = candidates[:top_k]
        
        results = []
        for idx, sim in top_candidates:
            product = self.products[idx]
            results.append({
                "id": product["id"],
                "name": product["name"],
                "price": product["price"],
                "category": product["category"],
                "similarity": sim
            })
        
        return results

# Initialize search engine
search_engine = ProductSearchEngine()

# Add products
products = [
    {
        "id": "p1",
        "name": "Wireless Noise-Canceling Headphones",
        "description": "Premium over-ear headphones with active noise cancellation and 30-hour battery",
        "category": "Electronics",
        "price": 299.99,
        "image_url": "https://example.com/headphones.jpg",
        "attributes": {"color": "black", "wireless": "yes", "battery_life": "30h"}
    },
    {
        "id": "p2",
        "name": "Running Shoes Pro",
        "description": "Lightweight athletic shoes with cushioned sole for marathon training",
        "category": "Sports",
        "price": 149.99,
        "image_url": "https://example.com/shoes.jpg",
        "attributes": {"color": "blue", "size_range": "7-13", "material": "mesh"}
    },
    {
        "id": "p3",
        "name": "Organic Coffee Beans",
        "description": "Single-origin Ethiopian coffee with notes of blueberry and chocolate",
        "category": "Food",
        "price": 24.99,
        "image_url": "https://example.com/coffee.jpg",
        "attributes": {"weight": "1lb", "roast": "medium", "organic": "yes"}
    },
    {
        "id": "p4",
        "name": "Smart Fitness Watch",
        "description": "GPS-enabled fitness tracker with heart rate monitor and sleep tracking",
        "category": "Electronics",
        "price": 199.99,
        "image_url": "https://example.com/watch.jpg",
        "attributes": {"color": "silver", "waterproof": "yes", "gps": "yes"}
    }
]

for p in products:
    search_engine.add_product(
        p["id"], p["name"], p["description"], p["category"],
        p["price"], p["image_url"], p["attributes"]
    )

# Test searches
print("ECOMMERCE PRODUCT SEARCH\n")
print("="*60 + "\n")

search_queries = [
    ("workout gear", None),
    ("something for my morning routine", None),
    ("tech gadgets", {"max_price": 250}),
    ("audio equipment", {"category": "Electronics"})
]

for query, filters in search_queries:
    print(f"Query: '{query}'")
    if filters:
        print(f"Filters: {filters}")
    results = search_engine.search(query, filters=filters, top_k=3)
    print("Results:")
    for r in results:
        print(f"  - {r['name']} (${r['price']}) - match: {r['similarity']:.3f}")
    print()

## ❌ Failure Case

In [ ]:
# Failure case: Limitations of cross-modal retrieval
print("CROSS-MODAL RETRIEVAL CHALLENGES\n")
print("="*60 + "\n")

challenges = [
    {
        "challenge": "Semantic Gap",
        "description": "Visual and textual semantics don't always align",
        "example": "Query 'beautiful' matches different images for different users",
        "mitigation": "Use user feedback and personalization"
    },
    {
        "challenge": "Fine-Grained Details",
        "description": "Specific visual attributes hard to capture in text",
        "example": "Query 'red dress with floral pattern' may miss variations",
        "mitigation": "Use attribute-based filtering alongside semantic search"
    },
    {
        "challenge": "Abstract Concepts",
        "description": "Abstract queries don't translate well to visual content",
        "example": "Query 'freedom' or 'happiness' returns inconsistent results",
        "mitigation": "Map abstract concepts to concrete visual representations"
    },
    {
        "challenge": "Ambiguity",
        "description": "Words with multiple meanings cause retrieval errors",
        "example": "Query 'apple' could mean fruit or company",
        "mitigation": "Use context and disambiguation techniques"
    }
]

for c in challenges:
    print(f"⚠️  {c['challenge']}")
    print(f"   Description: {c['description']}")
    print(f"   Example: {c['example']}")
    print(f"   Mitigation: {c['mitigation']}\n")

print("="*60)
print("IMPROVING RETRIEVAL QUALITY:")
print("="*60)
print("""
1. Use CLIP embeddings for better visual-text alignment
2. Implement re-ranking with cross-encoders
3. Add user feedback loops
4. Combine with traditional keyword search
5. Use query expansion and refinement
""")

## 📊 Benchmark Comparison

| Model | Text-to-Image | Image-to-Text | Training Data | Model Size |
|-------|---------------|---------------|---------------|------------|
| CLIP | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ | 400M pairs | 150M-400M |
| ALIGN | ⭐⭐⭐⭐ | ⭐⭐⭐⭐ | 1.8B pairs | 800M |
| BLIP-2 | ⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ | 129M pairs | 3.8B |
| OpenAI Embeddings | ⭐⭐⭐ | N/A | Proprietary | Unknown |

### Performance Metrics (Recall@K):
- **CLIP**: R@1=59.6%, R@5=80.4%, R@10=87.1%
- **ALIGN**: R@1=58.8%, R@5=79.1%, R@10=85.9%
- **BLIP-2**: R@1=61.2%, R@5=82.1%, R@10=88.3%

### Recommendations:
- **CLIP**: Best general-purpose, well-supported
- **BLIP-2**: Best for captioning + retrieval
- **OpenAI**: Simple integration, no hosting needed

## 🎮 Interactive Playground

In [ ]:
def cross_modal_playground():
    """Interactive cross-modal retrieval playground."""
    print("\n" + "="*60)
    print("CROSS-MODAL RETRIEVAL PLAYGROUND")
    print("="*60 + "\n")
    
    # Create fresh retriever
    playground_retriever = SimpleCrossModalRetriever()
    
    # Add diverse items
    items = [
        {
            "id": "1",
            "text": "Modern glass skyscraper reflecting blue sky in downtown business district",
            "image_url": "https://images.unsplash.com/photo-1486406146926-c627a92ad1ab?w=400"
        },
        {
            "id": "2",
            "text": "Cozy living room with fireplace, comfortable sofa, and warm lighting",
            "image_url": "https://images.unsplash.com/photo-1554995207-c18c203602cb?w=400"
        },
        {
            "id": "3",
            "text": "Fresh sushi platter with salmon, tuna, and various rolls on wooden board",
            "image_url": "https://images.unsplash.com/photo-1579871494447-9811cf80d66c?w=400"
        },
        {
            "id": "4",
            "text": "Mountain hiking trail with pine trees and misty morning atmosphere",
            "image_url": "https://images.unsplash.com/photo-1551632811-561732d1e306?w=400"
        },
        {
            "id": "5",
            "text": "Vintage typewriter on wooden desk with coffee cup and notebooks",
            "image_url": "https://images.unsplash.com/photo-1516979187457-637abb4f9353?w=400"
        },
        {
            "id": "6",
            "text": "Colorful hot air balloons floating over scenic valley at sunrise",
            "image_url": "https://images.unsplash.com/photo-1507608616759-54f48f0af0ee?w=400"
        }
    ]
    
    for item in items:
        playground_retriever.add_item(item["id"], item["text"], item["image_url"])
    
    print(f"Loaded {len(items)} items into the search index.\n")
    
    # Interactive search loop
    while True:
        print("\n" + "-"*60)
        query = input("\nEnter search query (or 'quit' to exit): ").strip()
        
        if query.lower() == 'quit':
            break
        
        if not query:
            continue
        
        print(f"\nSearching for: '{query}'\n")
        results = playground_retriever.text_search(query, top_k=3)
        
        print("Top Results:")
        for i, r in enumerate(results, 1):
            print(f"\n{i}. Similarity: {r['similarity']:.3f}")
            print(f"   Description: {r['text']}")
            print(f"   Image URL: {r['image_url']}")

cross_modal_playground()

## 💡 Tips & Tricks

### Embedding Strategies:
1. **Use CLIP for images**: Better visual-text alignment
2. **Enrich text descriptions**: Include attributes and context
3. **Normalize embeddings**: Ensures fair similarity comparison
4. **Dimensionality**: Balance between accuracy and efficiency

### Query Optimization:
- **Be specific**: Detailed queries yield better results
- **Use synonyms**: Expand queries with related terms
- **Query rewriting**: Use LLM to improve search queries
- **Hybrid search**: Combine semantic + keyword matching

### Performance Tips:
- **Index partitioning**: Split large indexes by category
- **Approximate search**: Use ANN for speed (FAISS, HNSW)
- **Caching**: Store popular query results
- **Batch processing**: Embed multiple items at once

## 📚 References

1. [CLIP: Learning Transferable Visual Models](https://arxiv.org/abs/2103.00020)
2. [ALIGN: Scaling Up Visual and Vision-Language Representation](https://arxiv.org/abs/2102.05918)
3. [BLIP-2: Bootstrapping Language-Image Pre-training](https://arxiv.org/abs/2301.12597)
4. [FAISS: A Library for Efficient Similarity Search](https://github.com/facebookresearch/faiss)
5. [Sentence Transformers for Multi-Modal](https://www.sbert.net/examples/applications/image-search/README.html)